# Problema Direto Estacionário (*Hard*-PINN)

Este *notebook* irá trabalhar com o mesmo problema utilizado no *notebook* `01_direct_stationary_vanilla.ipynb`, mas com uma abordagem distinta: ***Hard-PINN***.

**Autor:** Edélio Gabriel Magalhães de Jesus.

---

> ### **Requisitos**
>
>- Ter conferido a discussão do *notebook* `01_direct_stationary_vanilla.ipynb`

---

## O que é a abordagem *hard* nas PINN?

As PINN clássicas - chamadas de *vanilla-PINN* por Baty [[ref]](#hands-on-paper) - trabalham com as condições de contorno impondo via $L_{data}$. Isso, na prática, apenas "incentiva" que a rede respeite as CCs. A ideia, baseada em uma abordagem proposta por Lagaris, na década de 1990 [[ref]](#lagaris-paper), é forçar as CCs, introduzindo funções auxiliares na própria construção da rede, tal como se segue:

$$
u_{\theta} = A(x, y) + B(x, y)u^*_{\theta}(x, y)
$$

onde:

- $u^*_{\theta}(x, y)$ é a saída bruta da rede neural;
- $A(x, y)$ é a função que satisfaz exatamente as condições de contorno, sem parâmetros ajustáveis;
- $B(x, y)$ é a funçaõ de peso, que se anula em toda a fronteira $\delta \Omega$, sendo $\Omega$ o domínio do problema.

## O grande benefício

Como as CCs são satisfeitas por construção, $L_data$ desaparece completamente. A funçaõ de perda se reduz a:

$$
L(\theta) = L_{PDE}(\theta)
$$

## A grande desvantagem
**``"Com grandes poderes vem grades responsabilidades" - Tio Ben``**

A construção das funções $A$ e $B$ nem sempre é fácil. Na verdade, dificilmente será fácil. Ela depende fortemente da geometria e do tipo de condições de contorno do problema. Para CCs de Neumann ou domínios irregulares, por exemplo, encontrar essas funções pode ser muito complicado e até mesmo impossível - em tempo ábil -, sendo mais eficiente optar pela abordagem clássica. 

Em suma, se você tem um problema que, com muita criatividade, consegue construir $A$ e $B$ sem muitos percalções, vale a pena tentar.

## Construindo as funções auxiliares para o nosso problema

Para o nosso problemas, ficamos com:

$$
A(x,y)=y\cdot \sin{(\pi x)}$$

dado que nossas CCs são:

$$
u(x,0)=0\\
u(x,1)=\sin{(\pi x})\\
u(0,y)=0\\
u(1,y)=0
$$

E, para $B(x)$:

$$
B(x,y)=x(1-x)\cdot(1-y)
$$

A verificação fica a cargo do leitor.

## Aplicando a *Hard*-PINN

O código completo está localizado na pasta `scripts`, especificamente no arquivo `ex01_pinn_direct_stacionary_hard.py`. Para facilitar a discussão, colocarei apenas trechos necessários para uma compreensão mais aprofundada.

---

A célula seguinte serve para:

- Recarregar automaticamente qualquer arquivo que for editado nos scripts
- Encontrar a pasta dos *scripts*, permitindo importar as funções criadas

In [1]:
%load_ext autoreload
%autoreload 2

import sys
import os

sys.path.append(os.path.abspath("../scripts"))

import plotly.io as pio
pio.renderers.default = "plotly_mimetype"

### **Importações necessárias**

In [2]:
import torch.nn as nn
import torch.optim as optim
import torch
import plotly.graph_objects as go
from geral_functions import PINN, sample_collocation_rectangular, sample_boundary_rectangular_stationary
from ex02_pinn_direct_stationary_hard import train_hard, evaluate_hard
from ex01_pinn_direct_stacionary_vannila import analytical_solution
from plot_utils import plot_points_stationary, plot_loss, plot_heatmaps, plot_profiles


### **Parâmetros fixos do problema**

Vamos usar os mesmos parâmetros que utilizamos com a abordagem *vanilla*, para permitir uma comparação justa.

In [3]:
# Definição do local onde o código serpa executado. Por padrão, gpu
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Usando: {DEVICE}')

# Arquitetura da rede
N_INPUTS = 2
N_OUTPUTS = 1
N_HIDDEN = 16
N_LAYERS = 4
ACTIVATION = nn.Tanh

# Parâmetros do problema
LB = [0,0]
UB = [1,1]

BC_FNS = {
    'bottom': lambda x: torch.zeros_like(x),
    'top':    lambda x: torch.sin(torch.pi * x),
    'left':   lambda x: torch.zeros_like(x),
    'right':  lambda x: torch.zeros_like(x)
}

# Parâmetros de amostragem
N_COLLOC = 500
N_BC = 30

# Parâmetros do treinamento
W_DATA = 1.0
W_PDE = 1.0
N_EPOCHS = 50000
LR = 1e-4

Usando: cpu


### **Instanciando o modelo**

In [4]:
model = PINN(N_INPUTS, N_OUTPUTS, N_HIDDEN, N_LAYERS, ACTIVATION)

### **Amostragem dos pontos**

In [5]:
X_BC, U_BC = sample_boundary_rectangular_stationary(N_BC, LB, UB, BC_FNS, DEVICE)
X_COLLOC = sample_collocation_rectangular(N_COLLOC, LB, UB, DEVICE)

### **Instanciando o otimizador**

In [6]:
OPTIMIZER = torch.optim.Adam(model.parameters(), lr=LR)

### **Treinamento**

In [7]:
history = train_hard(model, OPTIMIZER, N_COLLOC, LB, UB, N_EPOCHS, DEVICE)

Epoch 00000 | Loss PDE: 1.70e+01
Epoch 00100 | Loss PDE: 1.65e+01
Epoch 00200 | Loss PDE: 1.43e+01
Epoch 00300 | Loss PDE: 1.05e+01
Epoch 00400 | Loss PDE: 9.24e+00
Epoch 00500 | Loss PDE: 9.50e+00
Epoch 00600 | Loss PDE: 7.31e+00
Epoch 00700 | Loss PDE: 6.34e+00
Epoch 00800 | Loss PDE: 6.78e+00
Epoch 00900 | Loss PDE: 6.52e+00
Epoch 01000 | Loss PDE: 5.86e+00
Epoch 01100 | Loss PDE: 5.78e+00
Epoch 01200 | Loss PDE: 5.67e+00
Epoch 01300 | Loss PDE: 5.59e+00
Epoch 01400 | Loss PDE: 5.23e+00
Epoch 01500 | Loss PDE: 4.66e+00
Epoch 01600 | Loss PDE: 4.55e+00
Epoch 01700 | Loss PDE: 4.29e+00
Epoch 01800 | Loss PDE: 4.35e+00
Epoch 01900 | Loss PDE: 3.24e+00
Epoch 02000 | Loss PDE: 3.87e+00
Epoch 02100 | Loss PDE: 4.02e+00
Epoch 02200 | Loss PDE: 3.52e+00
Epoch 02300 | Loss PDE: 3.33e+00
Epoch 02400 | Loss PDE: 2.84e+00
Epoch 02500 | Loss PDE: 2.50e+00
Epoch 02600 | Loss PDE: 2.66e+00
Epoch 02700 | Loss PDE: 2.19e+00
Epoch 02800 | Loss PDE: 2.45e+00
Epoch 02900 | Loss PDE: 1.80e+00
Epoch 0300

### **Visualizando os resultados do treinamento**

Agora, relembre que nossa função de perda mudou, ficando somente com a perda do reísduo da PDE:

$$
Loss = L_{PDE}
$$

que no código é simplesmente:

---
```python
[...]
    residual = pde_residual_hard(model, X_col)
    loss_pde = torch.mean(residual ** 2)

return loss_pde
    
```
---

É válido ressaltar, no entando, que o resíduo é calculado sobre a nossa função de tentativa, aquela composta pelas funções auziliares. No código, fizemos isso da seguinte forma:

---
```python
def pde_residual_hard(model, X):
    u = trial_solution(model, X)

    grads = torch.autograd.grad(
        outputs=u,
        inputs=X,
        grad_outputs=torch.ones_like(u),
        create_graph=True
    )[0]

    u_x = grads[:, 0].unsqueeze(1)
    u_y = grads[:, 1].unsqueeze(1)

    u_xx = torch.autograd.grad(
        outputs=u_x,
        inputs=X,
        grad_outputs=torch.ones_like(u_x),
        create_graph=True
    )[0][:, 0].unsqueeze(1)

    u_yy = torch.autograd.grad(
        outputs=u_y,
        inputs=X,
        grad_outputs=torch.ones_like(u_y),
        create_graph=True
    )[0][:, 1].unsqueeze(1)

    return u_xx + u_yy
```
---

onde ***trial_solution*** contém nossa função adaptada:

---
```python
[...]
    x = X[:, 0].unsqueeze(1)
    y = X[:, 1].unsqueeze(1)

    N = model(X)
    A = y * torch.sin(torch.pi * x)
    B = x * (1 - x) * y * (1 - y)

return A + B * N
```
---

Vamos olhar para a distribuição dos pontos.

In [8]:
plot_points_stationary(X_COLLOC)

Ressalto, novamente, que com essa abordagem, não se faz necessário ter pontos amostrados sob as condições de contorno.

Agora, vamos conferir a evolução da perda ao longo das épocas.

In [9]:
plot_loss(history)

Note que a perda da PDE alcançou valores da ordem de $10^{-6}$, de forma mais consistente que a versão *Vanilla*. Isso era esperado, dado que o modelo foi condicionado a respeitar de forma rígida às condições de contorno.

Vamos partir para a avalação do modelo.

### **Validando o modelo**

A validação também é feita sobre a função de tentativa:

---
```python

x = np.linspace(0, 1, n_grid)
y = np.linspace(0, 1, n_grid)
X, Y = np.meshgrid(x, y)

X_flat = torch.tensor(
    np.stack([X.ravel(), Y.ravel()], axis=1),
    dtype=torch.float32,
    device=device
)

with torch.no_grad():
    U_pred = trial_solution(X_flat).cpu().numpy().reshape(n_grid, n_grid)
    U_ref  = analytical_fn(X_flat).cpu().numpy().reshape(n_grid, n_grid)

```
---

In [10]:
results = evaluate_hard(model, analytical_solution, DEVICE)

In [11]:
plot_heatmaps(results['U_pred'], results['U_ref'], results['x'], results['y'])

In [12]:
plot_profiles(results['U_pred_slices'], results['U_ref_slices'], results['x'], ylabel='u(x,y)')

É nítida como o modelo *Hard* capturou muito bem a solução do nosso problema, superando o criado pela abordagem clássica, principalmente quando olhamos para o *heatmap* do erro absoluto: os maiores valores ainda estavam na escala de $10^{-4}$, enquanto que para o caso *Vanilla* ficaram em $10^3$.

---

> Note que, até o momento, trabalhamos com um problema estacionário, ou seja, apenas com valores de contorno. Mas existem outro tipo de problema, que consideram uma evolução temporal, a partir de Condições Iniciais.
>
> Caso tenha ficado interessado, é recomendado conferir os *notebooks* `03_direct_transient.ipynb` e `05_inverse_transient.ipynb`.

---

## Referências

<a id="hands-on-paper"> [1] </a> [BATY, Hubert. A hands-on introduction to physics-informed neural networks for solving partial differential equations with benchmark tests taken from astrophysics and plasma physics. arXiv preprint arXiv:2403.00599, 2024.](https://arxiv.org/html/2403.00599v1#S2)

<a id="lagaris-paper"> [2] </a> [Lagaris, I. E., Likas, A., & Fotiadis, D. I. (1998). Artificial neural networks for solving ordinary and partial differential equations. *IEEE Transactions on Neural Networks*, 9(5), 987–1000.](https://ieeexplore.ieee.org/abstract/document/712178?casa_token=2rA_zJmQZqMAAAAA:NxOK7I7O3K2uCIg_bzu0OSH-M8b4TMjcpXPYccEezGa-kDQCt7tkLV3tt1IKlmCkovMAetGhcdk)